# eye_cnn.ipynb — CNN 눈 뜸/감음 학습

- 데이터: **MRL Eye Dataset** (Kaggle `akashshingha850/mrl-eye-dataset`) — 이 데이터만 사용
- 1번 구간: MRL 데이터 확인 · 사람 단위 분할 / 2번 구간: 학습 → `eye_cnn.pt`
- 전처리는 서버와 같은 `sleep_detector_api/services/eye_preprocess.py` 의 `prep_eye` 를 쓴다
- 🔴 `kaggle.json` · MRL 사진 · `eye_cnn.pt` 는 **구글 드라이브에만** (저장소 · zip · 채팅에 올리지 않음)
- 🔴 GitHub 에 올리기 전: Colab → 수정 → **모든 출력 지우기** (MRL 사진이 출력에 남으면 안 됨)

## 1. MRL 데이터 확인 (학습 전에 먼저)

확인한 결과는 1-13 표에 정리했다.

| # | 확인할 것 | 셀 |
|---|---|---|
| 1 | 장수 84,898 · 이름 규칙 · 중복 | 1-3 |
| 2 | 폴더 라벨(awake/sleepy) = 파일 이름 눈 상태(1/0) — **전부** | 1-4 |
| 3 | Kaggle 기존 train/val/test 에 같은 사람이 섞였는지 — **전부** | 1-5 |
| 4 | 피험자별 장수 · 한 클래스만 있는 사람 · 안경 · 센서 | 1-6 |
| 5 | 이미지 크기 · 흑백 여부 · 가로세로 비 | 1-7 |
| 6 | 눈으로 보기 — 뜸/감음 · 센서 · 안경 · 반사 · 가장 작은/큰 사진 | 1-8 |
| 7 | **눈이 얼마나 꽉 차게 잘렸나** → 서버에서 자를 여유(margin) 정하기 | 1-9 |
| 8 | 64×64 흑백으로 줄였을 때 밝기 평균 · 표준편차 → 정규화 값 후보 | 1-10 |
| 9 | 사람 단위 분할 + 감음/뜸 비율 균형 | 1-11 |

- 결과 그림 · 분할표는 `OUT_DIR`(드라이브)에 저장된다

### 1-1. 드라이브 연결 + Kaggle 키

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
KAGGLE_JSON = '/content/drive/MyDrive/keys/kaggle.json'        # ← 본인 kaggle.json 위치
OUT_DIR = '/content/drive/MyDrive/sleep_detector/mrl_check'    # ← 결과 그림 · 분할표 저장 폴더
os.makedirs(OUT_DIR, exist_ok=True)

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy(KAGGLE_JSON, os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('키 준비 완료 (내용은 출력하지 않음)')

### 1-2. 내려받기 (Colab 로컬 디스크, 약 330MB — 드라이브에 풀지 않는다)

In [ ]:
%%bash
if [ ! -d /content/mrl ]; then
  pip -q install kaggle
  kaggle datasets download -d akashshingha850/mrl-eye-dataset -p /content/mrl --unzip -q
fi
echo "폴더:"; find /content/mrl -maxdepth 3 -type d | head -20
echo "PNG 수:"; find /content/mrl -name '*.png' | wc -l

### 1-3. 파일 이름 읽기 — 장수 · 이름 규칙 · 중복

In [ ]:
import glob, re, collections
from pathlib import Path
import pandas as pd

# s0001_00001_0_0_0_0_0_01 = 피험자_번호_성별_안경_눈상태_반사_조명_센서
NAME = re.compile(r'^(s\d{4})_(\d+)_([01])_([01])_([01])_([012])_([01])_(\d{2})$')
paths = sorted(glob.glob('/content/mrl/**/*.png', recursive=True))
rows, bad = [], []
for p in paths:
    stem = os.path.basename(p)[:-4]
    m = NAME.match(stem)
    if not m:
        bad.append(p)
        continue
    parts = set(Path(p).parts)
    rows.append(dict(path=p, name=stem, subject=m[1], img_no=int(m[2]), gender=int(m[3]), glasses=int(m[4]),
                     eye=int(m[5]), reflection=int(m[6]), light=int(m[7]), sensor=m[8],
                     kaggle_split=next((s for s in ('train', 'val', 'test') if s in parts), '?'),
                     folder=next((s for s in ('awake', 'sleepy') if s in parts), '?')))
df = pd.DataFrame(rows)

print('PNG 전체      :', len(paths))
print('이름 규칙 통과 :', len(df), '/ 불통과:', len(bad), bad[:5])
print('84,898 장인가  :', len(df) == 84898)
print('같은 이름 중복 :', int(df['name'].duplicated().sum()))
print('피험자 수      :', df['subject'].nunique())

### 1-4. 폴더 라벨 = 파일 이름 눈 상태? (전부)

- 기대: `sleepy` = 0(감음), `awake` = 1(뜸), 불일치 0장

In [ ]:
df['folder_eye'] = df['folder'].map({'sleepy': 0, 'awake': 1})
mismatch = df[df['folder_eye'] != df['eye']]
print('폴더 라벨 ≠ 파일 이름 눈 상태:', len(mismatch), '장')
display(pd.crosstab(df['folder'], df['eye'], margins=True))
display(mismatch.head(10)[['name', 'kaggle_split', 'folder', 'eye']])

### 1-5. Kaggle 기존 분할에 같은 사람이 섞였나? (전부)

- 섞였으면 기존 train/val/test 는 쓰지 않고 1-11 의 사람 단위 분할을 쓴다

In [ ]:
subj_split = df.groupby('subject')['kaggle_split'].agg(lambda s: '+'.join(sorted(set(s))))
print(subj_split.value_counts().to_string())
print('\n여러 split 에 걸친 피험자:', int((subj_split.str.contains('\\+')).sum()), '/', len(subj_split))
display(pd.crosstab(df['subject'], df['kaggle_split']).head(10))

### 1-6. 피험자별 장수 · 클래스 · 안경 · 센서

In [ ]:
per = (df.pivot_table(index='subject', columns='eye', values='name', aggfunc='count', fill_value=0)
         .rename(columns={0: 'closed', 1: 'open'}))
per['total'] = per['closed'] + per['open']
per['closed_ratio'] = (per['closed'] / per['total']).round(2)
per['glasses_ratio'] = df.groupby('subject')['glasses'].mean().round(2)
per['gender'] = df.groupby('subject')['gender'].agg(lambda s: s.mode()[0])
per['sensors'] = df.groupby('subject')['sensor'].agg(lambda s: ','.join(sorted(set(s))))
display(per.sort_values('total'))

print('한 클래스만 있는 피험자:', list(per[(per['closed'] == 0) | (per['open'] == 0)].index))
print('장수 최소 / 최대:', per['total'].idxmin(), per['total'].min(), '/', per['total'].idxmax(), per['total'].max())
for col in ['gender', 'glasses', 'eye', 'reflection', 'light', 'sensor']:
    print(f'{col:10s}', df[col].value_counts().sort_index().to_dict())

### 1-7. 이미지 크기 · 흑백 여부 (전부, 머리 부분만 읽어서 빠름)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sizes, modes = [], collections.Counter()
for p in df['path']:
    with Image.open(p) as im:
        sizes.append(im.size)
        modes[im.mode] += 1
df['w'] = [s[0] for s in sizes]
df['h'] = [s[1] for s in sizes]
df['aspect'] = (df['w'] / df['h']).round(2)

print('이미지 모드 (L = 흑백):', dict(modes))
display(df[['w', 'h', 'aspect']].describe().round(1))
print('센서별 중간값'); display(df.groupby('sensor')[['w', 'h', 'aspect']].median())
print('뜸/감음별 중간값'); display(df.groupby('eye')[['w', 'h', 'aspect']].median())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(df['w'], bins=60); ax[0].set_title('width (px)')
ax[1].hist(df['aspect'], bins=60); ax[1].set_title('width / height')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/size_hist.png', dpi=100); plt.show()

### 1-8. 눈으로 보기

- 제목: `피험자 e눈상태 g안경 / 가로x세로 s센서` — 라벨이 틀려 보이는 사진, 반쯤 감은 사진이 있는지 본다

In [ ]:
def grid(sub, title, fname, n=24, cols=8, seed=0):
    sub = sub.sample(min(n, len(sub)), random_state=seed)
    nrows = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(nrows, cols, figsize=(cols * 1.6, nrows * 1.9))
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, sub.iterrows()):
        ax.imshow(Image.open(r.path), cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'{r.subject} e{r.eye} g{r.glasses}\n{r.w}x{r.h} s{r.sensor}', fontsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{fname}', dpi=100); plt.show()

grid(df[df.eye == 1], 'open (eye=1)', 'grid_open.png')
grid(df[df.eye == 0], 'closed (eye=0)', 'grid_closed.png')
for s in sorted(df['sensor'].unique()):
    grid(df[df.sensor == s], f'sensor {s}', f'grid_sensor{s}.png', n=16)
grid(df[df.glasses == 1], 'glasses', 'grid_glasses.png', n=16)
grid(df[df.reflection == 2], 'big reflection', 'grid_reflection.png', n=16)
grid(df[df.light == 0], 'bad light', 'grid_badlight.png', n=16)
grid(df.nsmallest(16, 'w'), 'smallest', 'grid_smallest.png', n=16)
grid(df.nlargest(16, 'w'), 'largest', 'grid_largest.png', n=16)

### 1-9. 눈이 얼마나 꽉 차게 잘렸나 (서버 자르기 여유 정하기)

- 빨간 선 = 사진 가로 · 세로의 10% 간격. 뜬 눈 16장
- 이 값으로 서버의 **MediaPipe 눈 점 → 자르기 범위**를 정한다 (학습과 서버가 같은 모양이어야 함)

In [ ]:
sub = df[df.eye == 1].sample(16, random_state=1)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, (_, r) in zip(axes.flat, sub.iterrows()):
    ax.imshow(Image.open(r.path).convert('L').resize((130, 130)), cmap='gray', vmin=0, vmax=255)
    for t in range(0, 131, 13):
        ax.axvline(t, color='r', lw=0.8, alpha=0.6); ax.axhline(t, color='r', lw=0.8, alpha=0.6)
    ax.set_xlim(0, 129); ax.set_ylim(129, 0)  # 선 때문에 그림 밖에 흰 여백이 생기지 않게
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'{r.subject} {r.w}x{r.h}', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/crop_tightness.png', dpi=100); plt.show()

**1-9 결과 (1-8 · 1-9 그림을 눈으로 본 값 — 1-10b 평균 사진으로 다시 확인)**

- 사진은 **전부 정사각형** (가로세로 비 1.0, 84,898장 모두)
- 눈 가로 폭(눈꼬리~눈꼬리): 사진 가로의 약 **55~70%** (센서 02 의 s0031 은 약 35~40% 로 눈이 작게 찍힘)
- 눈은 가로 가운데, 세로는 가운데보다 **약간 아래** (눈 중심이 위에서 약 50~55%)
- 눈썹: **들어 있는 사진이 많음** (위쪽 20~30% 안), 사진마다 다름 → 자르는 범위가 사진마다 조금씩 다르다
- 라벨이 틀려 보이는 사진: 이번 표본에서는 못 찾음. 반쯤 감은 눈은 '뜸'(e1) 쪽에 일부 있음 (예: s0014 · s0019 의 가늘게 뜬 눈)

### 1-10. 64×64 흑백으로 줄였을 때 밝기 (정규화 값 후보)

- 피험자 · 눈 상태마다 최대 100장씩 표본
- 여기서는 측정용으로 PIL BILINEAR 로 줄인다. 학습(2번 구간) · 서버는 `prep_eye` (cv2 `INTER_AREA`) 를 쓴다

In [ ]:
import numpy as np
SIZE = 64  # 서버 config.CNN_INPUT_SIZE

samp = df.sample(frac=1, random_state=0).groupby(['subject', 'eye']).head(100)  # 피험자 · 눈 상태마다 최대 100장
arr = np.stack([np.asarray(Image.open(p).convert('L').resize((SIZE, SIZE), Image.BILINEAR), dtype=np.float32) / 255
                for p in samp['path']])
print(f'표본 {len(arr)}장 / 평균 {arr.mean():.4f} / 표준편차 {arr.std():.4f}  ← 정규화 값 후보')
samp = samp.assign(mean=arr.mean(axis=(1, 2)))
print('센서별 밝기'); display(samp.groupby('sensor')['mean'].describe().round(3))
print('뜸/감음별 밝기'); display(samp.groupby('eye')['mean'].describe().round(3))

### 1-10b. 평균 눈 사진 — 눈이 사진의 어디에, 얼마나 크게 있나 (1-9 를 숫자로 확인)

- 1-10 의 64×64 표본을 뜸 / 감음으로 나눠 평균. 어두운 띠 = 눈이 있는 자리
- 자르기 범위는 **뜬 눈 평균**으로 잰다. 사진 가장자리는 원래 어두워서(비네팅) 가장자리 15%는 빼고 찾는다
- 눈 폭 = 눈 높이의 가로 띠에서 '그 열의 평균보다 어두운 정도'가 가장 어두운 곳의 절반 이상인 구간

In [ ]:
def inner_argmin(p, lo=0.15, hi=0.85):
    a, b = int(len(p) * lo), int(len(p) * hi)
    return a + int(np.argmin(p[a:b]))

m_open = arr[(samp['eye'] == 1).to_numpy()].mean(axis=0)
m_closed = arr[(samp['eye'] == 0).to_numpy()].mean(axis=0)

# 눈 중심 = 가장자리 15% 를 뺀 안쪽에서 가장 어두운 점 (5×5 평균으로 부드럽게) — 가로줄 평균은 눈썹 띠에 끌려감
pad = np.pad(m_open, 2, mode='edge')
smooth = sum(pad[dy:dy + SIZE, dx:dx + SIZE] for dy in range(5) for dx in range(5)) / 25
a, b = int(SIZE * 0.15), int(SIZE * 0.85)
cy, cx = np.unravel_index(np.argmin(smooth[a:b, a:b]), (b - a, b - a))
cy, cx = int(cy) + a, int(cx) + a
dark = m_open[max(cy - 3, 0):cy + 4].mean(axis=0) - m_open.mean(axis=0)  # 눈 띠가 그 열 평균보다 얼마나 어두운가
def span(frac):
    level = dark[cx] * frac  # 가장 어두운 깊이의 frac 만큼 어두운 곳까지
    left = right = cx
    while left > 0 and dark[left - 1] < level:
        left -= 1
    while right < SIZE - 1 and dark[right + 1] < level:
        right += 1
    return left, right

left, right = span(0.5)
l20, r20 = span(0.2)
eye_w50, eye_w20 = (right - left + 1) / SIZE, (r20 - l20 + 1) / SIZE

print(f'뜬 눈 중심: 위에서 {cy / SIZE:.0%}, 왼쪽에서 {cx / SIZE:.0%}')
print(f'눈 폭 — 절반 깊이: 사진 가로의 {eye_w50:.0%} / 20% 깊이: {eye_w20:.0%}')
print('  (평균 사진은 흐려서 눈꼬리가 옅다 → MediaPipe 의 눈꼬리~눈꼬리 폭에는 20% 깊이 값이 더 가깝다. 1-9 눈 확인값 55~70% 와 비교)')
print(f'  → 자르기 한 변 S ≈ 눈 폭 W ÷ {eye_w20:.2f} = {1 / eye_w20:.2f}W  (20% 깊이 기준)')
print(f'자르기 중심 = 눈 중심보다 {cy / SIZE - 0.5:+.2f}S 아래에 눈이 오도록 (음수면 위)')
brow = inner_argmin(m_closed.mean(axis=1), 0.10, 0.45)
print(f'감은 눈 평균의 눈썹 띠: 위에서 {brow / SIZE:.0%}')

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, m, name in ((ax[0], m_open, 'open mean'), (ax[1], m_closed, 'closed mean')):
    a.imshow(m, cmap='gray')
    for t in np.linspace(0, SIZE - 1, 11):
        a.axvline(t, color='r', lw=0.8, alpha=0.6); a.axhline(t, color='r', lw=0.8, alpha=0.6)
    a.set_xlim(0, SIZE - 1); a.set_ylim(SIZE - 1, 0); a.set_title(name)
ax[0].axvspan(left, right, ymin=0.45, ymax=0.55, color='y', alpha=0.4)
ax[2].plot(np.arange(SIZE) / SIZE, m_open.mean(axis=1), label='open rows (top→bottom)')
ax[2].plot(np.arange(SIZE) / SIZE, dark, '--', label='open eye band - col mean (left→right)')
ax[2].plot(np.arange(SIZE) / SIZE, m_closed.mean(axis=1), label='closed rows (top→bottom)')
ax[2].axhline(dark[cx] * 0.5, color='gray', lw=0.6); ax[2].axhline(dark[cx] * 0.2, color='gray', lw=0.6, ls=':'); ax[2].set_xlabel('position (0 = top/left, 1 = bottom/right)'); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/mean_eye.png', dpi=100); plt.show()

### 1-11. 사람 단위 분할 (70 / 15 / 15) + 감음/뜸 비율 균형

1. 장수가 많은 사람부터 목표 비율에 가장 모자란 쪽에 배정
2. 피험자 두 명을 맞바꿔서 **장수 비율과 감음 비율이 목표에 더 가까워지면** 바꾼다 — 더 나아지지 않을 때까지 (같은 데이터면 항상 같은 결과)

In [ ]:
import itertools, json, datetime

TARGET = {'train': 0.70, 'val': 0.15, 'test': 0.15}
total = int(per['total'].sum())
overall_closed = per['closed'].sum() / total

split, cnt = {}, collections.Counter()
for s in per.sort_values('total', ascending=False).index:
    k = min(TARGET, key=lambda k: cnt[k] / total - TARGET[k])
    split[s] = k
    cnt[k] += per.loc[s, 'total']

def stats(split):
    out = {}
    for k in TARGET:
        ss = sorted(s for s in split if split[s] == k)
        n, c = int(per.loc[ss, 'total'].sum()), int(per.loc[ss, 'closed'].sum())
        out[k] = dict(subjects=ss, n_subjects=len(ss), images=n, ratio=round(n / total, 3),
                      closed=c, open=n - c, closed_ratio=round(c / n, 3))
    return out

def cost(split):
    st = stats(split)
    return sum(2 * abs(st[k]['ratio'] - TARGET[k]) + abs(st[k]['closed_ratio'] - overall_closed) for k in TARGET)

def show(title, st):
    print(title)
    for k, v in st.items():
        print(f"  {k:5s} {v['n_subjects']:2d}명 {v['images']:6d}장 ({v['ratio']:.1%})  감음 {v['closed_ratio']:.1%}")

show('처음 배정', stats(split))
while True:
    best_c, best_pair = cost(split), None
    for a, b in itertools.combinations(sorted(split), 2):
        if split[a] == split[b]:
            continue
        split[a], split[b] = split[b], split[a]
        c = cost(split)
        split[a], split[b] = split[b], split[a]
        if c < best_c - 1e-9:
            best_c, best_pair = c, (a, b)
    if best_pair is None:
        break
    a, b = best_pair
    split[a], split[b] = split[b], split[a]

final = stats(split)
show(f'\n균형 조정 후 (전체 감음 비율 {overall_closed:.1%})', final)
for k, v in final.items():
    print(f"{k}: {', '.join(v['subjects'])}")
one_class = list(per[(per['closed'] == 0) | (per['open'] == 0)].index)
print('\n한 클래스만 있는 피험자의 배정:', {s: split[s] for s in one_class})

json.dump({'made': str(datetime.date.today()), 'method': 'greedy + swap balance', 'split': split, 'summary': final},
          open(f'{OUT_DIR}/mrl_person_split.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
df['split'] = df['subject'].map(split)
print('\n저장:', f'{OUT_DIR}/mrl_person_split.json')

### 1-12. 결과 요약

In [ ]:
summary = {
    '장수': len(df), '84898과 같음': len(df) == 84898, '이름 불통과': len(bad), '이름 중복': int(df['name'].duplicated().sum()),
    '피험자': df['subject'].nunique(),
    '폴더 라벨 불일치': len(mismatch),
    'Kaggle 분할에서 여러 split 에 걸친 피험자': int(subj_split.str.contains('\\+').sum()),
    '한 클래스만 있는 피험자': one_class,
    '이미지 모드': dict(modes),
    '크기 중간값 (w, h)': (int(df['w'].median()), int(df['h'].median())),
    '크기 범위 w': (int(df['w'].min()), int(df['w'].max())),
    '64x64 밝기 평균 / 표준편차': (round(float(arr.mean()), 4), round(float(arr.std()), 4)),
    '분할': {k: f"{v['n_subjects']}명 {v['images']}장 감음 {v['closed_ratio']:.1%}" for k, v in final.items()},
}
for k, v in summary.items():
    print(f'- {k}: {v}')

### 1-13. 확인 결과 정리

| 확인 | 결과 | 판정 |
|---|---|---|
| 장수 · 이름 규칙 · 중복 | 84,898장, 이름 규칙 불통과 0, 중복 0, 피험자 37명 | ✅ |
| 폴더 라벨 = 파일 이름 눈 상태 | 불일치 **0장** (sleepy = 0 감음 41,946 / awake = 1 뜸 42,952) | ✅ |
| Kaggle 기존 분할 | **37명 전원**이 train · val · test 에 다 들어 있음 → 이미지 단위 무작위 분할 | ❌ 쓰지 않음 → 1-11 분할 사용 |
| 이미지 | 전부 흑백(L), 전부 정사각형, 52~299px (중간값 87) | ✅ |
| 센서 | 01 이 83% (70,142장). 센서마다 크기(01: 84 · 02: 126 · 03: 73px) · 밝기가 다름 | ⚠️ |
| **피험자별 감음 비율** | 0.02 ~ 1.00 으로 **극단적** (s0007 98% 감음, s0035 3% 감음) | ⚠️ 사람 단위 분할 + 비율 균형이 꼭 필요 |
| 한 클래스만 있는 피험자 | s0004 · s0008 (감음만) → 둘 다 test | ⚠️ test 의 감음 재현율이 두 사람에 영향받음 |
| 뜸 / 감음 사진 크기 | 감음 중간값 91px, 뜸 82px — 크기가 라벨과 관련 | ⚠️ 64×64 로 줄이면 크기 자체는 사라지지만 눈이 차지하는 비율 차이는 남을 수 있음 |
| 64×64 밝기 | 평균 0.3426, 표준편차 0.1024 | 참고 (정규화는 사진마다) |
| 분할 (1-11) | train 16명 59,560 / val 10명 12,724 / test 11명 12,614, **세 구간 모두 감음 49.4%** | ✅ 이 분할로 학습 |

**전처리 (`eye_preprocess.py` — 노트북과 서버가 같은 함수)**

| 단계 | 서버 (폰 얼굴 사진) | 학습 (MRL 눈 사진) |
|---|---|---|
| 자르기 `crop_eye` | MediaPipe 눈꼬리 점(33 · 133 / 362 · 263) 사이 폭 W → **한 변 S = 1.5W 정사각형**, 중심은 **눈 중심보다 0.03S 위** | 없음 (이미 눈만 잘림) |
| `prep_eye` | 흑백 → 64×64 (`cv2.INTER_AREA`) → **사진마다 평균 0 · 표준편차 1** | 같음 |

- 자르기 범위 근거 (1-10b): 뜬 눈 평균의 눈 중심이 위에서 53% · 왼쪽에서 50%, 눈 폭 66% → S ≈ 1.52W. 1-9 눈 확인값 55~70% (≈ 1.4~1.8W) 와 맞음. 감은 눈 평균의 눈썹 띠는 위에서 23%
- 사진마다 정규화하는 이유: MRL 은 적외선 흑백, 폰은 컬러라 전체 밝기 차이가 크다
- MRL 은 자른 범위가 사진마다 달라서 학습 증강에 크기 · 위치 흔들기를 넣는다 (2-4)

## 2. 학습

- **실행 순서:** 1-1 · 1-2 · 1-3 (드라이브 · 내려받기 · `df`) → **2-1 부터**. 1-4 ~ 1-12 는 다시 안 돌려도 됨 (분할은 1-11 이 저장한 `mrl_person_split.json` 을 읽음)
- **런타임:** 런타임 → 런타임 유형 변경 → **T4 GPU**
- **전처리는 `eye_preprocess.prep_eye` 하나만** — 노트북에서 `/255` · 정규화를 따로 하지 않음 (서버 `face_service.py` 와 같은 함수)
- **증강은 train 에만, 노트북에만** (원본 눈 사진 → 증강 → `prep_eye`). val · test 는 증강 없이 `prep_eye` 만
- **모델:** ResNet18 ImageNet 전이학습 (딥러닝 14차시). 흑백 1채널을 3채널로 복사, 출력 1개 = **감음 확률** (서버 `CNN_CLOSED_PROB` 0.5 와 비교)
- Colab 이 끊기면: 1-1 · 1-2 · 1-3 → 2-1 ~ 2-4 다시 실행 → 2-5 가 드라이브의 `last.pt` 에서 **이어서** 학습

### 2-1. 준비 — `eye_preprocess.py` 올리기 + 학습 설정

- 왼쪽 **파일 탭 → 업로드** 로 저장소의 `sleep_detector_api/services/eye_preprocess.py` 를 `/content` 에 올린다 (저장소가 비공개라 git clone 대신 업로드)
- 파일을 고쳤으면 다시 올리고 이 셀을 다시 실행

In [ ]:
import os, sys, time, json, datetime, importlib
import numpy as np, pandas as pd, cv2, torch

assert os.path.exists('/content/eye_preprocess.py'), '왼쪽 파일 탭에서 eye_preprocess.py 를 /content 에 올려 주세요'
sys.path.insert(0, '/content')
import eye_preprocess
importlib.reload(eye_preprocess)
from eye_preprocess import prep_eye

SIZE = 64            # 서버 config.CNN_INPUT_SIZE 와 같게
EPOCHS = 12          # 최대 (조기 종료 있음)
PATIENCE = 3         # val 손실이 이만큼 연속으로 안 좋아지면 멈춤
BATCH = 256
LR = 3e-4
PRETRAINED = True    # ImageNet 가중치 (14차시 전이학습)
NUM_WORKERS = 2      # Colab CPU 2개
SEED = 0

TRAIN_DIR = '/content/drive/MyDrive/sleep_detector/eye_cnn'   # 체크포인트 · eye_cnn.pt · 그림 (GitHub 에 안 올림)
os.makedirs(TRAIN_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '← GPU 아님: 런타임 유형을 T4 GPU 로')
print('torch', torch.__version__, '/ prep_eye 출력:', prep_eye(np.zeros((90, 90), np.uint8) + 128, SIZE).shape)

### 2-2. 사람 단위 분할 읽기 + 라벨

- 1-11 의 분할 (train 16명 / val 10명 / test 11명, 세 구간 감음 49.4%) 을 그대로 쓴다
- 라벨 `y` = **감음 1 / 뜸 0** (MRL 파일 이름의 눈 상태는 반대로 1 = 뜸)

In [ ]:
sp = json.load(open(f'{OUT_DIR}/mrl_person_split.json', encoding='utf-8'))
df['split'] = df['subject'].map(sp['split'])
assert df['split'].notna().all(), '분할 json 에 없는 피험자가 있음'
df['y'] = (1 - df['eye']).astype(np.float32)

print('분할 파일:', sp['made'], sp['method'])
display(df.groupby('split').agg(사람=('subject', 'nunique'), 장수=('y', 'size'), 감음비율=('y', 'mean')).round(3))

### 2-3. 사진 읽어 메모리에 두기 (약 1~2분)

- **원본 크기 그대로** 읽어 둔다 (52~299px, 약 0.7GB) — train 은 매번 원본에서 증강 → `prep_eye`, 서버가 `crop_eye` 결과를 `prep_eye` 에 넣는 것과 같은 순서
- val · test 는 `prep_eye` 결과를 한 번만 만들어 둔다

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def read_eye(p):
    img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    assert img is not None, p
    return img

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    IMGS = list(ex.map(read_eye, df['path'], chunksize=256))
Y = df['y'].to_numpy()
IDX = {k: np.flatnonzero(df['split'].to_numpy() == k) for k in ('train', 'val', 'test')}
PREP = {k: np.stack([prep_eye(IMGS[j], SIZE) for j in IDX[k]]) for k in ('val', 'test')}

print(f'{len(IMGS)}장 읽음 {time.time() - t0:.0f}초 / 원본 {sum(i.nbytes for i in IMGS) / 1e9:.2f}GB')
print({k: len(v) for k, v in IDX.items()}, '/ val 입력 평균 · 표준편차:', PREP['val'].mean().round(3), PREP['val'].std().round(3))

### 2-4. 증강 (train 만) + 데이터셋 + 모델

- **크기 ±15% · 위치 ±5% · 회전 ±10°** — 서버 `crop_eye` 범위가 사진마다 조금씩 다른 것 · MRL 도 사진마다 자른 범위가 다른 것 (1-9)
- **좌우 반전** — 왼눈 · 오른눈
- **밝기 · 대비 → 감마로:** `prep_eye` 가 사진마다 평균 0 · 표준편차 1 로 맞추기 때문에 '전체를 더 밝게 · 대비를 몇 배로' 같은 직선 변화는 **정규화하면 사라져서 효과가 없음**. 감마(곡선)는 남음
- **흐림 · 잡음** — 폰 사진의 흔들림 · 저화질

In [ ]:
import torchvision
import matplotlib.pyplot as plt

def augment(img, rng):
    """원본 눈 사진(uint8) → 증강한 눈 사진(uint8). 정규화는 하지 않음 (prep_eye 가 함)"""
    if img.ndim == 3:   # MRL 은 전부 흑백이지만 혹시 몰라서
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[2] == 3 else cv2.COLOR_BGRA2GRAY)
    h, w = img.shape
    if rng.random() < 0.5:
        img = np.ascontiguousarray(img[:, ::-1])
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-10, 10), rng.uniform(0.85, 1.15))
    M[:, 2] += rng.uniform(-0.05, 0.05, 2) * (w, h)
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    img = 255.0 * (img / 255.0) ** rng.uniform(0.6, 1.6)
    if rng.random() < 0.3:
        k = int(rng.choice([3, 5]))
        img = cv2.GaussianBlur(img, (k, k), 0)
    if rng.random() < 0.3:
        img = img + rng.normal(0, rng.uniform(2, 8), img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)


class EyeDS(torch.utils.data.Dataset):
    def __init__(self, split):
        self.split, self.idx = split, IDX[split]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        if self.split == 'train':
            x = prep_eye(augment(IMGS[self.idx[i]], np.random.default_rng()), SIZE)
        else:
            x = PREP[self.split][i]
        return torch.from_numpy(x)[None], torch.tensor(Y[self.idx[i]])


class EyeCNN(torch.nn.Module):
    """(N, 1, 64, 64) prep_eye 결과 → (N,) 감음 logit. sigmoid 하면 감음 확률"""
    def __init__(self, pretrained=True):
        super().__init__()
        self.net = torchvision.models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        self.net.fc = torch.nn.Linear(self.net.fc.in_features, 1)
    def forward(self, x):
        return self.net(x.expand(-1, 3, -1, -1)).squeeze(1)


loaders = {k: torch.utils.data.DataLoader(EyeDS(k), batch_size=BATCH, shuffle=(k == 'train'),
                                          num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                          persistent_workers=NUM_WORKERS > 0)
           for k in ('train', 'val', 'test')}

# 증강 눈으로 확인: 위 = 원본(prep_eye 만), 아래 = 증강 → prep_eye  (그림 글자는 영어 — Colab 글꼴에 한글 없음)
rng = np.random.default_rng(1)
pick = rng.choice(IDX['train'], 8, replace=False)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for c, j in enumerate(pick):
    axes[0, c].imshow(prep_eye(IMGS[j], SIZE), cmap='gray'); axes[0, c].set_title('closed' if Y[j] else 'open', fontsize=8)
    axes[1, c].imshow(prep_eye(augment(IMGS[j], rng), SIZE), cmap='gray')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/augment_check.png', dpi=80); plt.show()

### 2-5. 학습 (끊기면 이어서)

- 매 에포크 끝에 드라이브에 `last.pt` (이어 하기용) · `best_state.pt` (val 손실이 가장 낮은 가중치) 저장
- val 손실이 `PATIENCE` 번 연속 안 좋아지면 멈춤
- **처음부터 다시** 하려면 드라이브의 `last.pt` 를 지우고 실행

In [ ]:
model = EyeCNN(PRETRAINED).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))
loss_fn = torch.nn.BCEWithLogitsLoss()

LAST, BEST = f'{TRAIN_DIR}/last.pt', f'{TRAIN_DIR}/best_state.pt'
start, hist, best, bad = 0, [], float('inf'), 0
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, hist, best, bad = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
    print(f'이어서 학습: {start + 1}번째 에포크부터 (지금까지 최저 val 손실 {best:.4f})')


@torch.no_grad()
def predict(split):
    """감음 확률 (numpy)"""
    model.eval()
    out = []
    for x, _ in loaders[split]:
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(model(x.to(device, non_blocking=True)).float()).cpu())
    return torch.cat(out).numpy()


def bce(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


for ep in range(start, EPOCHS):
    if bad >= PATIENCE:
        print('조기 종료'); break
    t0 = time.time(); model.train()
    tl, n = 0.0, 0
    for x, y in loaders['train']:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device, enabled=(device == 'cuda')):
            loss = loss_fn(model(x).float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item() * len(y); n += len(y)
    sched.step()

    p, yv = predict('val'), Y[IDX['val']]
    vl = bce(p, yv)
    pred = p >= 0.5
    rec_c, rec_o = pred[yv == 1].mean(), (~pred[yv == 0]).mean()
    row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), val_acc=round(float((pred == yv).mean()), 4),
               val_closed_recall=round(float(rec_c), 4), val_open_recall=round(float(rec_o), 4),
               sec=round(time.time() - t0))
    hist.append(row); print(row)
    if vl < best:
        best, bad = vl, 0
        torch.save(model.net.state_dict(), BEST)
    else:
        bad += 1
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                    epoch=ep, hist=hist, best=best, bad=bad), LAST)

hist_df = pd.DataFrame(hist); display(hist_df)
hist_df.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', figsize=(6, 3)); plt.savefig(f'{TRAIN_DIR}/loss.png', dpi=80); plt.show()

### 2-6. test 평가 (가장 좋았던 가중치)

- 기준 0.5 (서버 `CNN_CLOSED_PROB`) 에서 정확도 · 감음 재현율(졸음을 놓치지 않는가) · 뜸 재현율(괜히 경고하지 않는가)
- **애매한 확률 0.4~0.6 비율** — 이 구간은 서버에서 '경고'로 다룸
- 사람별 · 센서별 · 안경별 정확도 — s0004 · s0008 (감음만 있는 사람) 은 test 에 있음
- ⚠️ 여기 숫자는 **MRL(적외선)** 기준. 진짜 성능은 팀 사진(KPI ①)으로 따로 잰다

In [ ]:
model.net.load_state_dict(torch.load(BEST, map_location=device, weights_only=True))
p_test, y_test = predict('test'), Y[IDX['test']]
t = df.iloc[IDX['test']].copy()
t['p'], t['pred'] = p_test, (p_test >= 0.5).astype(int)
t['ok'] = (t['pred'] == t['y']).astype(int)

cm = pd.crosstab(t['y'].map({1: '정답 감음', 0: '정답 뜸'}), t['pred'].map({1: '예측 감음', 0: '예측 뜸'}))
test_metrics = dict(
    acc=round(float(t['ok'].mean()), 4),
    closed_recall=round(float(t.loc[t.y == 1, 'ok'].mean()), 4),
    open_recall=round(float(t.loc[t.y == 0, 'ok'].mean()), 4),
    closed_precision=round(float(t.loc[t.pred == 1, 'y'].mean()), 4),
    loss=round(bce(p_test, y_test), 4),
    ambiguous_0_4_to_0_6=round(float(((t.p >= 0.4) & (t.p <= 0.6)).mean()), 4),
)
print(test_metrics); display(cm)

print('\n기준값별 (서버 CNN_CLOSED_PROB 후보)')
display(pd.DataFrame([dict(기준=th, 정확도=((p_test >= th) == y_test).mean(), 감음재현율=(p_test[y_test == 1] >= th).mean(),
                           뜸재현율=(p_test[y_test == 0] < th).mean()) for th in (0.3, 0.4, 0.5, 0.6, 0.7)]).round(4))

by_subject = t.groupby('subject').agg(장수=('ok', 'size'), 감음비율=('y', 'mean'), 정확도=('ok', 'mean')).round(3)
display(by_subject.sort_values('정확도'))
for col in ('sensor', 'glasses', 'reflection'):
    display(t.groupby(col).agg(장수=('ok', 'size'), 정확도=('ok', 'mean')).round(3))

# 틀린 사진 보기 (확신하고 틀린 것부터)
wrong = t[t.ok == 0].assign(conf=lambda d: (d.p - 0.5).abs()).sort_values('conf', ascending=False).head(16)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax in axes.flat:
    ax.axis('off')
for ax, (_, r) in zip(axes.flat, wrong.iterrows()):
    ax.imshow(IMGS[r.name], cmap='gray')
    ax.set_title(f"{r.subject} {'closed' if r.y else 'open'} p={r.p:.2f}", fontsize=7)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/test_wrong.png', dpi=80); plt.show()

### 2-7. CPU 속도 (서버는 GPU 없음)

- 사진 1장: `prep_eye` + CNN, CPU 스레드 1개 / 전체

In [ ]:
cpu_model = EyeCNN(pretrained=False)
cpu_model.net.load_state_dict(torch.load(BEST, map_location='cpu', weights_only=True))
cpu_model.eval()
sample = IMGS[IDX['test'][0]]

def one(img):
    with torch.no_grad():
        return torch.sigmoid(cpu_model(torch.from_numpy(prep_eye(img, SIZE))[None, None])).item()

latency = {}
for th in (1, torch.get_num_threads()):
    torch.set_num_threads(th)
    for _ in range(5):
        one(sample)
    t0 = time.perf_counter()
    for _ in range(50):
        one(sample)
    latency[f'{th}스레드_ms'] = round((time.perf_counter() - t0) / 50 * 1000, 2)
print('사진 1장 (눈 하나):', latency, '— 서버는 눈 2개라 약 2배')

### 2-8. `eye_cnn.pt` 저장 (드라이브 → 서버 `ml_models/`, GitHub 에 안 올림)

- `state_dict` 는 **torchvision `resnet18` 그대로의 이름** (서버에서 클래스를 따로 안 만들어도 됨, 아래 2-9)
- 50MB 를 넘으면 안 됨 (ResNet18 은 약 45MB)

In [ ]:
PT = f'{TRAIN_DIR}/eye_cnn.pt'
torch.save(dict(
    state_dict=torch.load(BEST, map_location='cpu', weights_only=True),
    arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
    output='logit 1개, sigmoid = 감음 확률',
    prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
    crop='eye_preprocess.crop_eye (S = 1.5W, 중심은 눈 중심보다 0.03S 위)',
    data='MRL Eye Dataset, 사람 단위 분할 (mrl_person_split.json)',
    epochs=len(hist), best_val_loss=round(best, 4), test=test_metrics, cpu_latency=latency,
    made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),   # str 로: weights_only 로 읽을 수 있게
), PT)
mb = os.path.getsize(PT) / 1e6
print(f'저장: {PT}  {mb:.1f}MB', '✅' if mb < 50 else '❌ 50MB 넘음')

# 저장한 파일을 다시 읽어서 같은 값이 나오는지 (서버에서 읽는 방법 그대로)
ck = torch.load(PT, map_location='cpu', weights_only=True)
m2 = torchvision.models.resnet18(weights=None); m2.fc = torch.nn.Linear(512, 1)
m2.load_state_dict(ck['state_dict']); m2.eval()
with torch.no_grad():
    x = torch.from_numpy(PREP['test'][:64])[:, None]
    same = torch.allclose(torch.sigmoid(m2(x.expand(-1, 3, -1, -1)).squeeze(1)), torch.sigmoid(cpu_model(x)), atol=1e-5)
print('다시 읽어도 같은 결과:', same)

### 2-9. 서버에서 쓰는 법 (A · `face_service.py` 참고용 — 서버 코드는 A 담당)

```python
import torch, torchvision
import config
from services.eye_preprocess import crop_eye, prep_eye

def load_eye_cnn():
    ck = torch.load(config.CNN_MODEL_PATH, map_location="cpu", weights_only=True)
    m = torchvision.models.resnet18(weights=None)
    m.fc = torch.nn.Linear(m.fc.in_features, 1)
    m.load_state_dict(ck["state_dict"])
    return m.eval()

@torch.no_grad()
def closed_prob(model, image, eye_points):
    """얼굴 사진(BGR) + 한 눈의 MediaPipe 눈 점 → 감음 확률 (0~1)"""
    x = torch.from_numpy(prep_eye(crop_eye(image, eye_points), config.CNN_INPUT_SIZE))[None, None]
    return torch.sigmoid(model(x.expand(-1, 3, -1, -1))).item()
```

- 두 눈의 확률 평균 ≥ `CNN_CLOSED_PROB`(0.5) → 감음. 0.4~0.6 은 애매 → '경고' 쪽으로
- `requirements.txt` 의 `torch` · `torchvision` (CPU 판) 주석을 풀어야 함 (CNN 을 쓸 때)

### 2-10. 학습 결과 (Colab T4, 에포크당 약 37초)

| 항목 | 값 |
|---|---|
| 에포크 (조기 종료) | 4 에포크에서 멈춤 — **1 에포크 가중치가 최고** (그 뒤 train 손실만 줄고 val 손실은 늘어남) |
| 최저 val 손실 | 0.0986 (val 정확도 96.9%) |
| test 정확도 | **95.9%** (12,614장, 11명) |
| test 감음 재현율 / 뜸 재현율 | **97.5%** / 94.4% (감음 정밀도 94.5%) — 졸음을 놓치는 쪽보다 괜히 감음이라 하는 쪽이 조금 많음 |
| 기준값별 | 0.3: 감음 98.5% · 뜸 91.3% / **0.5: 97.5% · 94.4%** / 0.7: 96.4% · 97.0% |
| 애매한 확률(0.4~0.6) 비율 | 1.9% |
| 가장 낮은 사람 · 센서 | s0022 77.5%, s0028 90.2% (저화질) / 센서 01 95.8% · 02 98.6%, 안경 95.1% · 안경 없음 96.2% |
| 틀린 사진 (2-6 그림) | 안경 반사로 눈이 안 보이는 '뜸' 사진을 감음으로 · 가늘게 뜬 듯한 '감음' 사진을 뜸으로 (MRL 라벨 자체가 애매한 사진) |
| CPU 속도 (눈 1개, 1스레드) | Colab 13.3ms (개발 PC 약 5ms) → 두 눈 약 10~30ms |
| `eye_cnn.pt` 크기 | 44.8MB (50MB 안) · `weights_only=True` 로 다시 읽기 확인 |

- ⚠️ 모두 MRL(적외선) test 기준. 폰 컬러 사진 성능은 팀 사진(KPI ①)으로 따로 잰다

- 3번 구간 비교의 'MRL만' 은 **이 모델 (MRL test 95.9%)**. 참고: 10.27 다른 GPU 런타임에서 2번 구간을 다시 학습했을 때는 MRL test 96.9% 였지만, AI-Hub 에서는 이 모델과 **반대 방향으로** 틀림 (3-7 참고)

## 3. AI-Hub 173 미세조정 (한국인 · 실제 차량)

- **왜:** MRL 은 외국인 · 적외선 · 실험실 데이터라 MRL test 95.9% 가 우리 앱 성능을 뜻하지 않는다 (강사님 피드백). 한국인 운전자 데이터로 다시 재고, 미세조정한다
- **데이터:** AI-Hub 「졸음운전 예방을 위한 운전자 상태 정보 영상」 (dataSetSn=173) — bbox **실제도로환경** (버스 · 승용 · 택시 · 트럭) + **통제환경**
- **자르기는 서버와 똑같이:** 얼굴 사진 → MediaPipe 눈 점 → `crop_eye` → `prep_eye`. JSON 눈 상자로 자르지 않는다 (상자는 MediaPipe 가 같은 눈을 찾았는지 확인용으로만). **라벨만 JSON 에서** (`Leye` · `Reye` 의 `Opened`)
- MediaPipe 가 얼굴을 못 찾은 사진은 버린다 (서버에서도 CNN 까지 가지 않음)
- **운전자 단위 분할**, test 는 **실제 도로 운전자만**
- 비교할 모델 3개: **MRL만** (`eye_cnn.pt`, 2번 구간) / **미세조정** (MRL → AI-Hub, 학습률 1/10) / **AI-Hub만** (ImageNet 에서 시작) × MRL test · AI-Hub test. 팀 사진 열은 KPI ① (`kpi_eye_accuracy.ipynb`)
- 🔴 **재배포 금지:** AI-Hub 원본 · 자른 눈 사진(`.npz`) 은 **본인 드라이브에만**. GitHub · 단톡 · 공유 드라이브 X. 숫자 · 학습된 모델은 공유 가능
- 🔴 GitHub 에 올리기 전 **모든 출력 지우기** (3-1 눈 사진 그림)

**라벨 JSON 확인 결과 (10.27, Validation 라벨 2개 — PC 에서 숫자만 확인)**

| 확인 | 실제도로환경 | 통제환경 |
|---|---|---|
| JSON 수 · 운전자 | 23,514 · 63명 (버스 23 · 승용 23 · 택시 8 · 트럭 9) | 12,563 · 27명 |
| 폴더 · 파일 이름 | `1.버스/R_237_40_M/R_237_40_M_20_M0_G0_C0_02.json` (R_번호_나이_성별_장면_마스크_안경_담배_프레임) | `045_G1/045_G1_01_<조명>_<행동>_<상태>_<날짜>_<시각>_<프레임>.json` |
| 운전자 번호 | 폴더 번호 = `UserInfo.ID` (불일치 0) | 폴더 앞 번호 = `UserInfo.ID` |
| 사진 | 720×1280, `Channel` 1 | 800×1280, `Channel` 1 |
| `Leye` 가 사진 왼쪽인가 | 99.98% → **`Leye` = 사진 왼쪽 눈 = 서버 `LEFT_EYE`(33 …)** | 99.85% |
| 감음 (두 눈 다) | 약 8% (운전자별 1~42%) | 약 28% |
| 한쪽만 감음 | 4.2% → 눈 하나씩 라벨을 따로 쓴다 | 5.2% |

- 감음이 적다 (실제 도로) → train 에서 감음 · 뜸을 **반반 뽑는다** (MRL 학습과 같은 비율, 서버 기준 0.5 유지). 평가는 감음 재현율 · 뜸 재현율을 따로 본다
- `Channel` 1 → 원본도 흑백(적외선 카메라일 수 있음). 3-1 그림으로 확인

**받기 · 자르기는 PC 에서** (`notebooks/aihub_crop.py`): AI-Hub 는 해외 IP 다운로드를 막아서 Colab 에서 못 받는다 (HTTP 502)
1. PC: `sleep_detector_api\venv\Scripts\python.exe notebooks\aihub_crop.py download` → `crop` (서버와 같은 mediapipe 1.1.0 · `crop_eye`)
2. PC 의 `D:\aihub173\eye_npz\` 폴더 안 파일을 드라이브 `sleep_detector/aihub_eye/` 에 올린다 (본인 드라이브만)
3. Colab: 1-1 · 1-2 · 1-3 → 2-1 ~ 2-4 (MRL test · `augment` · `EyeCNN`) → **3-1 부터**

### 3-1. 자른 눈 사진 읽기 (PC 의 `aihub_crop.py` 결과)

- 드라이브 `sleep_detector/aihub_eye/` 의 `crops_*.npz` + `summary.json`
- 눈 하나 = 한 줄: 96×96 흑백 눈 사진 + 라벨 (`y` 감음 1 / 뜸 0) · 운전자 번호 · `env` (road 실제도로 / ctrl 통제) · 차종 · 안경 · 마스크 · 성별 · 나이(10살 단위)
- PC 에서 이미 버린 것: 얼굴 못 찾음 · 안 보이는 눈 · MediaPipe 눈 중심이 JSON 눈 상자(50% 넓힘) 밖
- 그림은 눈만 (얼굴은 안 보여줌). GitHub 올리기 전에 출력 지우기

In [ ]:
import collections, glob
CACHE_DIR = '/content/drive/MyDrive/sleep_detector/aihub_eye'   # 본인 드라이브에만
SUMMARY = json.load(open(f'{CACHE_DIR}/summary.json', encoding='utf-8'))
missing = [n for n in SUMMARY['chunks'] if not os.path.exists(f'{CACHE_DIR}/{n}')]
assert not missing, f'드라이브에 없는 파일 {len(missing)}개: {missing[:3]}'

parts = [np.load(f'{CACHE_DIR}/{n}') for n in SUMMARY['chunks']]
CROPS = np.concatenate([z['X'] for z in parts])
meta = pd.DataFrame({k: np.concatenate([z[k] for z in parts]) for k in ('fname', 'side', 'y', 'driver', 'env', 'cat', 'src', 'glasses', 'mask', 'gender', 'age')})
meta = meta.rename(columns={'fname': 'file'})
meta['gender'] = meta['gender'].map({1: '남', 2: '여'})   # JSON UserInfo.Gender (1 남 · 2 여)
AY = meta['y'].to_numpy(np.float32)
MAX_PER_DRIVER = SUMMARY['max_per_driver']

c = SUMMARY['counts']
print(f"PC 에서 만든 날: {SUMMARY['made']} / 사진 {c['files']}장 → 얼굴 없음 {c.get('noface', 0)} ({c.get('noface', 0) / c['files']:.1%})"
      f" · 안 보이는 눈 {c.get('eye_not_visible', 0)} · 상자 밖 {c.get('outside_box', 0)} → 쓰는 눈 {len(meta)}")
print(f'메모리 {CROPS.nbytes / 1e9:.2f}GB')
display(meta.groupby(['src', 'env', 'cat']).agg(운전자=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3))
both = sorted(set(meta[meta.env == 'road'].driver) & set(meta[meta.env == 'ctrl'].driver))
print('실제도로 · 통제 양쪽에 있는 운전자 번호 (같은 사람으로 봄):', both)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for row, yv in enumerate((0, 1)):
    for ax, j in zip(axes[row], np.random.default_rng(0).choice(np.flatnonzero(AY == yv), 8, replace=False)):
        ax.imshow(CROPS[j], cmap='gray'); ax.set_title(f"{'closed' if yv else 'open'} {meta.side[j]} {meta.env[j]}", fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

### 3-2. 운전자 단위 분할 — test 는 실제 도로 운전자만

- 목표 장수 비율 train 70 / val 15 / test 15 (MRL 1-11 과 같은 욕심쟁이 배정 + 감음 비율 맞추기)
- test 후보 = 실제도로에만 있는 운전자. 통제환경 운전자는 train · val 에만
- 분할표는 드라이브 `aihub_person_split.json` (같은 데이터면 항상 같은 결과)

In [ ]:
per_a = meta.groupby('driver').agg(n=('y', 'size'), closed=('y', 'sum'), envs=('env', lambda s: '+'.join(sorted(set(s)))))
total_a, closed_all = per_a.n.sum(), per_a.closed.sum() / per_a.n.sum()
TGT = {'train': 0.70, 'val': 0.15, 'test': 0.15}
asplit, acnt = {}, collections.Counter()


def score(k, d):   # 이 운전자를 k 에 넣었을 때: 장수가 목표보다 얼마나 넘치나 + 감음 비율이 전체와 얼마나 다른가 (작을수록 좋음)
    n, c = acnt[k] + per_a.n[d], acnt[k + '_c'] + per_a.closed[d]
    return n / total_a - TGT[k] + 0.3 * abs(c / n - closed_all)


for d in per_a.sort_values('n', ascending=False).index:
    allowed = ['train', 'val', 'test'] if per_a.envs[d] == 'road' else ['train', 'val']
    k = min(allowed, key=lambda k: score(k, d))
    asplit[int(d)] = k; acnt[k] += per_a.n[d]; acnt[k + '_c'] += per_a.closed[d]

meta['split'] = meta.driver.map(asplit)
AIDX = {k: np.flatnonzero(meta.split.to_numpy() == k) for k in TGT}
display(meta.groupby('split').agg(운전자=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean'),
                                  실제도로비율=('env', lambda s: (s == 'road').mean())).round(3))
display(meta[meta.split == 'test'].groupby('cat').agg(운전자=('driver', 'nunique'), 눈=('y', 'size'), 감음비율=('y', 'mean')).round(3))
json.dump({'made': str(datetime.date.today()), 'max_per_driver': MAX_PER_DRIVER, 'npz_made': SUMMARY['made'], 'split': asplit},
          open(f'{CACHE_DIR}/aihub_person_split.json', 'w', encoding='utf-8'), indent=1)
APREP = {k: np.stack([prep_eye(CROPS[j], SIZE) for j in AIDX[k]]) for k in ('val', 'test')}

### 3-3. 학습 함수 — 미세조정 (MRL → AI-Hub) · AI-Hub만

- train: `augment`(2-4 와 같음) → `prep_eye`, 감음 · 뜸을 **반반 뽑기** (`WeightedRandomSampler`)
- **미세조정:** `eye_cnn.pt` 가중치에서 시작, 학습률 `LR / 10` (3e-5), 최대 5 에포크, val 손실 2번 연속 안 좋아지면 멈춤
- **AI-Hub만:** ImageNet 에서 시작, 2번 구간과 같은 설정 (`LR` · `EPOCHS` · `PATIENCE`)
- 끊기면 다시 실행 → 드라이브의 `aihub_<이름>_last.pt` 에서 이어서

In [ ]:
class AiDS(torch.utils.data.Dataset):
    def __init__(self, split):
        self.split, self.idx = split, AIDX[split]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = prep_eye(augment(CROPS[j], np.random.default_rng()), SIZE) if self.split == 'train' else APREP[self.split][i]
        return torch.from_numpy(x)[None], torch.tensor(AY[j])


ytr = AY[AIDX['train']]
w = np.where(ytr == 1, 0.5 / ytr.mean(), 0.5 / (1 - ytr.mean()))
aloader = {'train': torch.utils.data.DataLoader(AiDS('train'), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
                                                sampler=torch.utils.data.WeightedRandomSampler(w, len(w), replacement=True))}
for k in ('val', 'test'):
    aloader[k] = torch.utils.data.DataLoader(AiDS(k), batch_size=BATCH, num_workers=NUM_WORKERS)


@torch.no_grad()
def predict_arr(m, X, bs=512):
    """prep_eye 결과 (N, 64, 64) → 감음 확률"""
    m.eval(); out = []
    for i in range(0, len(X), bs):
        x = torch.from_numpy(X[i:i + bs])[:, None].to(device)
        with torch.autocast(device, enabled=(device == 'cuda')):
            out.append(torch.sigmoid(m(x).float()).cpu())
    return torch.cat(out).numpy()


def fit(name, init_state, lr, epochs, patience):
    m = EyeCNN(pretrained=init_state is None).to(device)
    if init_state is not None:
        m.net.load_state_dict(init_state)
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    last, best_p = f'{TRAIN_DIR}/aihub_{name}_last.pt', f'{TRAIN_DIR}/aihub_{name}_best.pt'
    start, h, best_l, bad_n = 0, [], float('inf'), 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        m.load_state_dict(ck['model']); o.load_state_dict(ck['opt']); s.load_state_dict(ck['sched'])
        start, h, best_l, bad_n = ck['epoch'] + 1, ck['hist'], ck['best'], ck['bad']
        print(f'{name}: {start + 1}번째 에포크부터 이어서')
    yv = AY[AIDX['val']]
    for ep in range(start, epochs):
        if bad_n >= patience:
            print('조기 종료'); break
        t0 = time.time(); m.train(); tl = n = 0
        for x, y in aloader['train']:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = loss_fn(m(x).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
            tl += loss.item() * len(y); n += len(y)
        s.step()
        p = predict_arr(m, APREP['val']); vl = bce(p, yv); pred = p >= 0.5
        row = dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), val_acc=round(float((pred == yv).mean()), 4),
                   val_closed_recall=round(float(pred[yv == 1].mean()), 4), val_open_recall=round(float((~pred[yv == 0]).mean()), 4),
                   sec=round(time.time() - t0))
        h.append(row); print(name, row)
        if vl < best_l:
            best_l, bad_n = vl, 0; torch.save(m.net.state_dict(), best_p)
        else:
            bad_n += 1
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=s.state_dict(), epoch=ep, hist=h, best=best_l, bad=bad_n), last)
    return torch.load(best_p, map_location='cpu', weights_only=True), h


loss_fn = torch.nn.BCEWithLogitsLoss()
STATE = {'MRL만': torch.load(f'{TRAIN_DIR}/eye_cnn.pt', map_location='cpu', weights_only=True)['state_dict']}
print('train 눈', len(AIDX['train']), '/ 감음', f'{ytr.mean():.1%}', '→ 반반 뽑기')

### 3-4. 학습 실행

In [ ]:
STATE['미세조정'], HIST_FT = fit('finetune', STATE['MRL만'], LR / 10, 5, 2)
STATE['AI-Hub만'], HIST_AI = fit('scratch', None, LR, EPOCHS, PATIENCE)

### 3-5. 세 모델 비교 (MRL test · AI-Hub test)

- 기준 0.5 (서버 `CNN_CLOSED_PROB`). AI-Hub test 는 감음이 적으므로 **균형 정확도** (감음 재현율과 뜸 재현율의 평균) 를 같이 본다
- AI-Hub test 는 차종별 (트럭 = 우리 대상) · 안경 · 마스크 · 성별 · 나이별로도 (마스크 쓴 운전자도 되는지)
- 틀린 눈 사진 16장 (미세조정 모델, 확신하고 틀린 것부터) — 가는 눈 · 반쯤 감은 눈이 어떻게 틀리는지
- test 운전자는 학습 · val 에 안 쓴 사람만 (3-2 운전자 단위 분할)

In [ ]:
def metrics(p, y, th=0.5):
    pr = p >= th
    rc, ro = float(pr[y == 1].mean()), float((~pr[y == 0]).mean())
    return dict(acc=round(float((pr == y).mean()), 4), balanced_acc=round((rc + ro) / 2, 4), closed_recall=round(rc, 4),
                open_recall=round(ro, 4), closed_precision=round(float(y[pr].mean()) if pr.any() else float('nan'), 4), n=int(len(y)))


ta = meta.iloc[AIDX['test']].copy()
rows_cmp, P_AI = [], {}
for k in ('MRL만', '미세조정', 'AI-Hub만'):
    m = EyeCNN(pretrained=False).to(device); m.net.load_state_dict(STATE[k])
    p_mrl, P_AI[k] = predict_arr(m, PREP['test']), predict_arr(m, APREP['test'])
    rows_cmp.append({'모델': k, **{f'MRL_{a}': b for a, b in metrics(p_mrl, Y[IDX['test']]).items()},
                     **{f'AIHub_{a}': b for a, b in metrics(P_AI[k], AY[AIDX['test']]).items()}})
cmp_df = pd.DataFrame(rows_cmp).set_index('모델')
display(cmp_df[['MRL_acc', 'MRL_closed_recall', 'MRL_open_recall', 'AIHub_acc', 'AIHub_balanced_acc', 'AIHub_closed_recall', 'AIHub_open_recall', 'AIHub_closed_precision']])

for col in ('cat', 'glasses', 'mask', 'gender', 'age'):
    display(pd.DataFrame({k: {g: metrics(P_AI[k][(ta[col] == g).to_numpy()], AY[AIDX['test']][(ta[col] == g).to_numpy()])['balanced_acc']
                              for g in sorted(ta[col].unique())} for k in P_AI}).rename_axis(col))

# 틀린 눈 사진 (미세조정, 확신하고 틀린 것부터) — 눈만, 얼굴 X. GitHub 올리기 전 출력 지우기
yt = AY[AIDX['test']]
wrong = np.flatnonzero((P_AI['미세조정'] >= 0.5) != (yt == 1))
wrong = wrong[np.argsort(-np.abs(P_AI['미세조정'][wrong] - 0.5))][:16]
print(f"틀린 눈 {int(((P_AI['미세조정'] >= 0.5) != (yt == 1)).sum())} / {len(yt)}")
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax in axes.flat:
    ax.axis('off')
for ax, i in zip(axes.flat, wrong):
    r = ta.iloc[i]
    ax.imshow(CROPS[AIDX['test'][i]], cmap='gray')
    ax.set_title(f"{'closed' if r.y else 'open'} p={P_AI['미세조정'][i]:.2f}\n{'glasses ' if r.glasses else ''}{'mask' if r['mask'] else ''}", fontsize=7)
plt.tight_layout(); plt.savefig(f'{TRAIN_DIR}/aihub_test_wrong.png', dpi=80); plt.show()

print('기준값별 (미세조정)')
display(pd.DataFrame([{'기준': th, **metrics(P_AI['미세조정'], AY[AIDX['test']], th)} for th in (0.3, 0.4, 0.5, 0.6, 0.7)]).set_index('기준'))
print('기준값별 (MRL만) — 같은 AI-Hub test')
display(pd.DataFrame([{'기준': th, **metrics(P_AI['MRL만'], AY[AIDX['test']], th)} for th in (0.3, 0.5, 0.7)]).set_index('기준'))

### 3-6. 저장 — `eye_cnn_aihub.pt` (미세조정) · 비교 숫자

- 형식은 `eye_cnn.pt` 와 같다 → 서버 코드 그대로, `config.CNN_MODEL_PATH` 파일만 바꾸면 됨 (채택 여부는 11/4 KPI ① 로 결정)
- AI-Hub만 모델은 `eye_cnn_aihub_only.pt`. 비교 숫자 `aihub_compare.json` 은 공유 가능 (사진 없음)

In [ ]:
for k, fname in (('미세조정', 'eye_cnn_aihub.pt'), ('AI-Hub만', 'eye_cnn_aihub_only.pt')):
    torch.save(dict(
        state_dict=STATE[k], arch='resnet18', in_channels='흑백 1채널 → 3채널 복사', input_size=SIZE,
        output='logit 1개, sigmoid = 감음 확률',
        prep='eye_preprocess.prep_eye (흑백 → size×size → 사진마다 평균 0 · 표준편차 1)',
        crop='MediaPipe 눈 점 → eye_preprocess.crop_eye (서버와 같음)',
        data=('MRL → ' if k == '미세조정' else '') + 'AI-Hub 173 졸음운전 예방을 위한 운전자 상태 정보 영상 (bbox 실제도로 + 통제), 운전자 단위 분할',
        test={'MRL': {a[4:]: float(v) for a, v in cmp_df.loc[k].items() if a.startswith('MRL_')},   # float: weights_only=True 로 읽히게
              'AIHub': {a[6:]: float(v) for a, v in cmp_df.loc[k].items() if a.startswith('AIHub_')}},
        made=str(datetime.datetime.now())[:16], torch=str(torch.__version__),
    ), f'{TRAIN_DIR}/{fname}')
    torch.load(f'{TRAIN_DIR}/{fname}', map_location='cpu', weights_only=True)  # 서버와 같은 방식으로 다시 읽기
    print('저장:', fname, f"{os.path.getsize(f'{TRAIN_DIR}/{fname}') / 1e6:.1f}MB", '· weights_only=True 로 읽힘')

json.dump({'made': str(datetime.datetime.now())[:16], 'compare': cmp_df.reset_index().to_dict('records'),
           'hist_finetune': HIST_FT, 'hist_aihub_only': HIST_AI,
           'test_drivers': int(ta.driver.nunique()), 'test_eyes': int(len(ta))},
          open(f'{TRAIN_DIR}/aihub_compare.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장: aihub_compare.json')

### 3-7. 결과 (10.27 Colab T4)

**AI-Hub test = 실제 도로 운전자만, 눈 33,190개 / 기준 0.5**

| 모델 | MRL test 정확도 | AI-Hub test 정확도 | AI-Hub 균형 정확도 | AI-Hub 감음 재현율 / 뜸 재현율 | 감음 정밀도 | 팀 사진 (KPI ①) |
|---|---|---|---|---|---|---|
| MRL만 (`eye_cnn.pt`) | **95.9%** | 76.2% | 82.5% | 91.7% / **73.2%** | **40.0%** | ? |
| **미세조정** (`eye_cnn_aihub.pt`) | **96.2%** | **95.2%** | **93.8%** | **91.7% / 95.9%** | **81.2%** | ? |
| AI-Hub만 (`eye_cnn_aihub_only.pt`) | 92.9% | 95.0% | 93.4% | 91.1% / 95.8% | 80.8% | ? |

- **MRL만은 한국인 실제 운전자의 뜬 눈 4개 중 1개 이상을 '감음'으로 봄** (뜸 재현율 73.2%) → '감음' 판정 중 실제 감은 눈은 40% 뿐 = **괜한 졸음 경고가 쏟아짐** → 강사님 지적을 숫자로 확인
- 같은 코드로 다른 런타임에서 다시 학습한 MRL 모델 (MRL test 96.9%) 은 반대로 **감은 눈을 3개 중 1개 놓침** (감음 재현율 66.5%, 뜸 94.0%, 균형 80.2%) → **MRL 만으로 학습하면 한국인 실제 운전자에서 어느 쪽으로 틀릴지조차 들쭉날쭉** (균형 정확도 80~82%)
- **미세조정: 균형 정확도 82.5% → 93.8%, 뜸 재현율 73.2% → 95.9%, 감음 정밀도 40.0% → 81.2%, 감음 재현율 91.7% 유지, MRL test 95.9% → 96.2% (오히려 조금 오름)**
- AI-Hub만은 AI-Hub 에서 비슷하지만 MRL test 92.9% 로 떨어짐 → **미세조정 모델을 씀**
- 학습: 미세조정 4 에포크 가중치 최고 (val 손실 0.1475, val 정확도 94.7%) / AI-Hub만 1 에포크 최고 (val 손실 0.1440) → 4 에포크에서 조기 종료. 에포크당 약 92~97초

**나눠 보기 (AI-Hub test 균형 정확도)**

| 구분 | MRL만 | 미세조정 | AI-Hub만 |
|---|---|---|---|
| 버스 · 승용 · 택시 | 82.6% · 83.2% · 75.7% | 93.1% · 95.2% · 89.8% | 93.0% · 94.8% · 89.4% |
| **트럭 (우리 대상)** | 86.0% | **96.5%** | 95.4% |
| 안경 안 씀 / 씀 | 83.7% / 80.6% | 95.3% / 91.3% | 94.9% / 90.9% |
| **마스크 안 씀 / 씀** | 82.5% / 82.3% | **94.1% / 93.2%** | 93.6% / 93.1% |
| 남 / 여 | 82.3% / 83.9% | 93.5% / 97.9% | 93.1% / 97.7% |
| 20 · 30 · 40 · 50 · 60 · 70대 | 68.4 · 81.0 · 84.4 · 82.2 · 83.8 · 78.2% | 94.7 · 93.4 · 95.0 · 93.4 · 93.0 · 87.5% | 95.0 · 93.0 · 94.1 · 92.9 · 93.4 · 90.0% |

- 마스크를 써도 (얼굴만 찾으면) 차이 1%p — 마스크는 눈 판정에 거의 영향 없음
- 안경이 가장 어려움 (약 4%p 낮음, 렌즈 반사 — MRL 때와 같은 경향). 70대는 사람 수가 적어 흔들릴 수 있음
- 틀린 눈 1,598 / 33,190 (4.8%)

**기준값별 (미세조정, 서버 `CNN_CLOSED_PROB` 후보)**

| 기준 | 감음 재현율 | 뜸 재현율 | 균형 정확도 | 감음 정밀도 |
|---|---|---|---|---|
| 0.3 | 94.1% | 93.9% | 94.0% | 75.0% |
| 0.4 | 93.2% | 95.1% | **94.1%** | 78.6% |
| **0.5** | 91.7% | 95.9% | 93.8% | 81.2% |
| 0.6 | 89.9% | 96.6% | 93.2% | 83.6% |
| 0.7 | 87.9% | 97.2% | 92.5% | 85.9% |

- 졸음을 놓치지 않는 쪽이면 0.3~0.4 (감은 눈 +1.5~2.4%p, 괜한 경고 +0.8~2%p). test 로 본 값이라 조금 낙관적 → **최종 기준은 팀 사진(KPI ①)으로**
- MRL만은 기준을 바꿔도 균형 정확도 77~84% (0.3: 감음 96.3% · 뜸 58.4% / 0.7: 감음 82.0% · 뜸 85.4%) — 기준값으로는 해결 안 됨

**⚠️ 한계**
- AI-Hub 도 **적외선** 영상 (눈동자 반사점 · `Channel` 1) → "외국인 → 한국인" 은 해결, "적외선 → 폰 카메라" 차이는 남음 → **11/4 채택 결정은 팀 사진(KPI ①) 점수로**
- 마스크 · 고개 돌림 사진은 MediaPipe 가 얼굴을 못 찾아 학습 · 평가 모두에서 빠짐 (서버도 같음 → 앱 판단대로 확정)
- 출처: AI-Hub 「졸음운전 예방을 위한 운전자 상태 정보 영상」 (한국지능정보사회진흥원). 원본 · 자른 눈 사진은 재배포하지 않음

## 4. 팀 사진 · 영상 프레임 CNN 확률 → CSV (KPI ① 용)

- **결과는 CSV 하나:** `cnn.csv` = `file, label, p_mrl, p_aihub` (파일 이름 · 정답 · MRL만 감음 확률 · 미세조정 감음 확률)
  - 비교 표 (ML Kit · EAR · CNN) 는 A 의 `kpi_eye_accuracy.ipynb` 가 **`file` 로 붙여서** 만든다 → 같은 표를 두 사람이 만들지 않게
- 계산은 **서버 `face_service` 와 똑같이**: 사진(`IMREAD_COLOR`) → MediaPipe 눈 점(픽셀) → `crop_eye` (None 이면 CNN 없음, 눈 크기는 폭 기준으로만 — 10/28 서버에서 8px 상자 검사 삭제) → `prep_eye` → **두 눈 평균** 감음 확률
  - CNN 없음 (얼굴 못 찾음 · 눈이 너무 작음 · 사진 밖) = 빈칸 → A 노트북에서 '값 없음'
- `label` = 파일 이름 속 `closed` / `half` / `open` (서버 `_label_of` 와 같은 규칙)
- 파일 이름은 **촬영 수칙** 그대로 — 사진 `p1_bright_open_s01.jpg` · 영상 프레임 `p1_bright_closed_v03.jpg` (A 의 `tools/kpi_frames.py` 가 만듦)
  - **사진 · 영상 프레임 둘 다** 넣는다 (10/28): 기준값은 영상 쪽 숫자로 정하므로, CNN 도 ML Kit · EAR 과 같은 프레임으로 잰다
  - 둘을 가르는 건 A 노트북이 파일 이름(`_v` / `_s`)으로 한다 → `cnn.csv` 칸은 그대로
- 🔴 팀 사진은 **CNN 학습에 쓰지 않음** (평가만). 사진 · CSV 는 드라이브에만, GitHub 에 올리기 전 출력 지우기
- **실행:** 이 구간만 따로 돌려도 됨 (1 ~ 3번 구간 불필요). `/content` 에 `eye_preprocess.py` (main 최신) 업로드 · GPU 필요 없음

### 4-1. 준비 — 경로 · MediaPipe · 모델 두 개

In [ ]:
import os, sys, glob, csv, collections, urllib.request
import numpy as np, cv2, torch, torchvision   # pandas 없이 (서버 venv 에서도 돌게)

try:                                   # Colab 이면 드라이브
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/sleep_detector'
    sys.path.insert(0, '/content')     # eye_preprocess.py 업로드 위치
    WORK = '/content'
except ImportError:                    # PC (서버 venv) 면 저장소 · D:\aihub173
    ROOT = os.environ.get('SD_ROOT', r'D:\aihub173')
    sys.path.insert(0, os.path.abspath(os.path.join('..', 'sleep_detector_api', 'services')))
    WORK = ROOT

IMAGE_DIRS = [f'{ROOT}/kpi_eye/photos',              # ← 팀 사진 (촬영 수칙 이름, _s번호)
              f'{ROOT}/kpi_eye/frames']              # ← 영상 프레임 (kpi_frames.py video 결과, _v번호) — 한 폴더에 같이 있어도 됨
OUT_CSV = f'{ROOT}/kpi_eye/cnn.csv'                   # ← A 노트북 DATA_DIR 과 같은 폴더
MODELS = {'p_mrl': f'{ROOT}/eye_cnn/eye_cnn.pt',      # MRL만
          'p_aihub': f'{ROOT}/eye_cnn/eye_cnn_aihub.pt'}  # 미세조정
if not os.path.exists(MODELS['p_mrl']):               # PC: D:\aihub173\models\
    MODELS = {k: f'{ROOT}/models/{os.path.basename(v)}' for k, v in MODELS.items()}

try:
    import mediapipe as mp
except ImportError:
    %pip -q install mediapipe==1.1.0
    import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from eye_preprocess import crop_eye, prep_eye

FACE_TASK = f'{WORK}/face_landmarker.task'
if not os.path.exists(FACE_TASK):
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task', FACE_TASK)
det = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_buffer=open(FACE_TASK, 'rb').read()), num_faces=1))

LEFT_EYE = [33, 160, 158, 133, 153, 144]     # face_service.py 와 같은 번호
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
SIZE = 64                                    # config.CNN_INPUT_SIZE


def load_cnn(path):                          # 서버 _get_cnn 과 같음
    ck = torch.load(path, map_location='cpu', weights_only=True)
    m = torchvision.models.resnet18(weights=None)
    m.fc = torch.nn.Linear(m.fc.in_features, 1)
    m.load_state_dict(ck['state_dict'])
    return m.eval()


NETS = {k: load_cnn(v) for k, v in MODELS.items()}
photos = {}                                  # 파일 이름 → 경로 (같은 이름이 두 폴더에 있으면 하나만)
for d in IMAGE_DIRS:
    for f in sorted(glob.glob(f'{d}/*')):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            photos.setdefault(os.path.basename(f), f)
photos = [photos[k] for k in sorted(photos)]
n_v = sum('_v' in os.path.basename(f).lower() for f in photos)
print('모델', {k: os.path.basename(v) for k, v in MODELS.items()}, '/ 사진', len(photos) - n_v, '장 · 영상 프레임', n_v, '장')
print('없는 폴더:', [d for d in IMAGE_DIRS if not os.path.isdir(d)])

### 4-2. 사진마다 두 모델 감음 확률 → `cnn.csv`

In [ ]:
def label_of(name):                          # 서버 _label_of 와 같음
    name = name.lower()
    return next((l for l in ('closed', 'half', 'open') if l in name), '')


@torch.no_grad()
def closed_probs(path):
    """사진 1장 → {'p_mrl': …, 'p_aihub': …} (서버 _cnn_closed_prob 과 같은 계산) / CNN 없음 → 빈 dict, 이유"""
    bgr = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)   # 서버 _read_image_bgr
    if bgr is None:
        return {}, '읽기 실패'
    h, w, _ = bgr.shape
    res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
    if not res.face_landmarks:
        return {}, '얼굴 없음'
    lm = res.face_landmarks[0]
    eyes = [np.array([(lm[i].x * w, lm[i].y * h) for i in idx]) for idx in (LEFT_EYE, RIGHT_EYE)]
    # 눈 크기는 crop_eye 의 '폭' 기준으로만 (서버 10/28: 눈 점 '높이' 8px 검사는 감은 눈에서만 CNN 이 꺼지는 버그라 삭제)
    crops = [crop_eye(bgr, pts) for pts in eyes]
    if any(c is None for c in crops):
        return {}, 'crop_eye None (눈이 너무 작음 · 사진 밖)'
    x = torch.from_numpy(np.stack([prep_eye(c, SIZE) for c in crops]))[:, None].expand(-1, 3, -1, -1)
    return {k: round(float(torch.sigmoid(m(x).squeeze(1)).mean()), 4) for k, m in NETS.items()}, ''


rows, why = [], collections.Counter()
for f in photos:
    probs, reason = closed_probs(f)
    name = os.path.basename(f)
    rows.append({'file': name, 'label': label_of(name), 'p_mrl': probs.get('p_mrl', ''), 'p_aihub': probs.get('p_aihub', '')})
    if reason:
        why[reason] += 1
with open(OUT_CSV, 'w', newline='', encoding='utf-8-sig') as fp:   # A 노트북 · 서버 result.csv 와 같은 인코딩
    wr = csv.DictWriter(fp, fieldnames=['file', 'label', 'p_mrl', 'p_aihub'])
    wr.writeheader(); wr.writerows(rows)
print('저장:', OUT_CSV, len(rows), '장')

# 확인만 (비교 표는 A 노트북에서): 라벨 없는 사진 · CNN 없는 사진 · 라벨별 확률 중간값
print('라벨 없는 사진:', [r['file'] for r in rows if not r['label']][:10])
print('CNN 없음:', sum(why.values()), '장', dict(why))
for src, is_v in (('사진', False), ('영상', True)):
    for lab in ('open', 'half', 'closed'):
        g = [r for r in rows if r['label'] == lab and r['p_mrl'] != '' and ('_v' in r['file'].lower()) == is_v]
        if g:
            print(f"{src} {lab:6s} {len(g):3d}장  감음 확률 중간값  MRL만 {np.median([r['p_mrl'] for r in g]):.3f} · 미세조정 {np.median([r['p_aihub'] for r in g]):.3f}")